In [ ]:
import math
import copy

SIZE = 3
AI = "X"
HUMAN = "O"
BLOCKED = "#"

DIRECTIONS = [(-1,0),(1,0),(0,-1),(0,1),
              (-1,-1),(-1,1),(1,-1),(1,1)]


class IsolationGame:
    def __init__(self):
        self.board = [[str(r*SIZE + c + 1) for c in range(SIZE)] for r in range(SIZE)]
        self.ai_pos = None
        self.human_pos = None

    def print_board(self):
        print("\nBoard:")
        for row in self.board:
            print(" | ".join(row))
        print()

    def cell_to_pos(self, cell):
        cell = int(cell) - 1
        return (cell // SIZE, cell % SIZE)

    def get_moves(self, pos):
        if pos is None:
            return [(r,c) for r in range(SIZE) for c in range(SIZE)
                    if self.board[r][c].isdigit()]

        moves = []
        r, c = pos
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            while 0 <= nr < SIZE and 0 <= nc < SIZE and self.board[nr][nc].isdigit():
                moves.append((nr, nc))
                nr += dr
                nc += dc
        return moves

    def apply_move(self, pos, move, player):
        new_game = copy.deepcopy(self)

        if pos is not None:
            r, c = pos
            new_game.board[r][c] = BLOCKED

        r, c = move
        new_game.board[r][c] = player

        if player == AI:
            new_game.ai_pos = move
        else:
            new_game.human_pos = move

        return new_game

    def is_terminal(self):
        if self.ai_pos is None or self.human_pos is None:
            return False
        return len(self.get_moves(self.ai_pos)) == 0 or len(self.get_moves(self.human_pos)) == 0

    def evaluate(self):
        return len(self.get_moves(self.ai_pos)) - len(self.get_moves(self.human_pos))


def minimax(game, depth, alpha, beta, maximizing):
    if depth == 0 or game.is_terminal():
        return game.evaluate(), None

    if maximizing:
        max_eval = -math.inf
        best_move = None
        for move in game.get_moves(game.ai_pos):
            new_game = game.apply_move(game.ai_pos, move, AI)
            eval_score, _ = minimax(new_game, depth-1, alpha, beta, False)

            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move

            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval, best_move

    else:
        min_eval = math.inf
        best_move = None
        for move in game.get_moves(game.human_pos):
            new_game = game.apply_move(game.human_pos, move, HUMAN)
            eval_score, _ = minimax(new_game, depth-1, alpha, beta, True)

            if eval_score < min_eval:
                min_eval = eval_score
                best_move = move

            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval, best_move

game = IsolationGame()
print("Welcome to Isolation Game (3x3)")
game.print_board()

while True:
    cell = input("Choose your starting cell (1-9): ")
    try:
        r, c = game.cell_to_pos(cell)
        if game.board[r][c].isdigit():
            game = game.apply_move(None, (r,c), HUMAN)
            break
        else:
            print("Cell already taken!")
    except:
        print("Enter a number between 1 and 9")

_, ai_move = minimax(game, 2, -math.inf, math.inf, True)
game = game.apply_move(None, ai_move, AI)

print("\nAfter initial placement:")
game.print_board()

while True:
    human_moves = game.get_moves(game.human_pos)
    if not human_moves:
        print("No moves left for Human. AI Wins!")
        break

    print("Your possible moves:", [game.board[r][c] for r,c in human_moves])

    cell = input("Enter your move (cell number): ")
    try:
        r, c = game.cell_to_pos(cell)
        if (r,c) not in human_moves:
            print("Invalid move!")
            continue
    except:
        print("Enter a valid number!")
        continue

    game = game.apply_move(game.human_pos, (r,c), HUMAN)
    game.print_board()

    ai_moves = game.get_moves(game.ai_pos)
    if not ai_moves:
        print("No moves left for AI. Human Wins!")
        break

    _, best_move = minimax(game, 4, -math.inf, math.inf, True)
    print("AI moves to cell:", game.board[best_move[0]][best_move[1]])
    game = game.apply_move(game.ai_pos, best_move, AI)
    game.print_board()


Welcome to Isolation Game (3x3)

Board:
1 | 2 | 3
4 | 5 | 6
7 | 8 | 9

Choose your starting cell (1-9): 5

After initial placement:

Board:
1 | X | 3
4 | O | 6
7 | 8 | 9

Your possible moves: ['8', '4', '6', '1', '3', '7', '9']
Enter your move (cell number): 8

Board:
1 | X | 3
4 | # | 6
7 | O | 9

AI moves to cell: 1

Board:
X | # | 3
4 | # | 6
7 | O | 9

Your possible moves: ['7', '9', '4', '6']
Enter your move (cell number): 9

Board:
X | # | 3
4 | # | 6
7 | # | O

AI moves to cell: 4

Board:
# | # | 3
X | # | 6
7 | # | O

Your possible moves: ['6', '3']
Enter your move (cell number): 6

Board:
# | # | 3
X | # | O
7 | # | #

AI moves to cell: 7

Board:
# | # | 3
# | # | O
X | # | #

Your possible moves: ['3']
Enter your move (cell number): 3

Board:
# | # | O
# | # | #
X | # | #

No moves left for AI. Human Wins!
